# Medical Coding Assist Agent

This is the flattened version of the Capstone project, fully self-contained for Kaggle submission.

In [1]:
# Install dependencies if running in a raw Kaggle environment
# !pip install chromadb pandas lxml requests google-genai kagglehub pydantic
import json
import os
import random
import time

from google import genai
from pydantic import BaseModel
import chromadb
import kagglehub
import logging
import pandas as pd
import requests
import shutil


# NOTE: Set your Gemini API key here or in Kaggle Secrets
# os.environ["GEMINI_API_KEY"] = "your_key"
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
try:
    client = genai.Client(api_key=GEMINI_API_KEY)
    print("Gemini initialized.")
except Exception as e:
    print("Gemini initialization failed:", e)

Gemini initialization failed: No API key was provided. Please pass a valid API key. Learn how to create an API key at https://ai.google.dev/gemini-api/docs/api-key.


In [2]:
print(os.getenv("github_token"))

None


In [ ]:
# Download the Patient Records Dataset
def download_patient_records():
    print("Downloading patient-records dataset from Kaggle...")
    path = kagglehub.dataset_download("sergionefedov/patient-records-100k-patients-15-conditions")
    print("Downloaded to:", path)
    
    # In a notebook environment, the data is already in `path`
    return path

# In Kaggle, we might already have the dataset attached, but we'll download it to be safe.
DATA_PATH = download_patient_records()

## Data Processing

In [ ]:
def load_patient_diagnoses(csv_path: str) -> pd.DataFrame:
    """
    Loads the diagnoses.csv file and processes the ICD-10 codes.
    
    The 'secondary_icd10s' column contains a pipe-delimited list of codes.
    This function parses the primary and secondary codes into a single list
    of all assigned codes per encounter/row.
    """
    df = pd.read_csv(csv_path)
    
    # Fill NaN values with empty strings for easier string manipulation
    df['primary_icd10'] = df['primary_icd10'].fillna('')
    df['secondary_icd10s'] = df['secondary_icd10s'].fillna('')
    
    def parse_codes(row):
        codes = []
        
        # 1. Add primary code
        primary = str(row['primary_icd10']).strip()
        if primary:
            codes.append(primary)
            
        # 2. Add secondary codes (split by pipe)
        secondary_str = str(row['secondary_icd10s']).strip()
        if secondary_str:
            # Split and clean the codes
            secondary = [c.strip() for c in secondary_str.split('|') if c.strip()]
            codes.extend(secondary)
            
        return codes

    # Apply the parsing function row by row
    df['assigned_icd10_codes'] = df.apply(parse_codes, axis=1)
    
    # We mainly care about patient_id, visit_date, and our parsed list of codes
    # Returning a subset of columns keeps memory usage low
    return df[['patient_id', 'visit_date', 'assigned_icd10_codes']]



## Prompts

In [ ]:
"""
Prompts for the Medical Coding Assist Agent.
These prompts configure the LLM to act as an expert coder, extract entities, and reconcile codes.
"""

SYSTEM_PROMPT = """
You are an expert Medical Coder and Auditor. 
Your primary goal is to accurately translate clinical documentation into ICD-10-CM diagnostic codes, 
strictly adhering to the ICD-10-CM Official Guidelines for Coding and Reporting.

You have access to the following tools via MCP:
- search_icd10: Search the ICD-10-CM guidelines and tabular list using a vector database.
- normalize_medical_term: Query the UMLS API to normalize medical jargon or symptoms.
- validate_icd10_codes: Check proposed codes against Excludes1, Excludes2, and 7th character rules.
- read_clinical_note: Securely read a clinical note from the dataset.

Always prioritize definitive diagnoses over symptoms, unless a definitive diagnosis has not been established.
Ensure all proposed codes are fully justified by both the clinical text and specific ICD-10-CM rules.
When a condition is documented without additional clinical information (e.g., acute vs. chronic, or specific anatomical site), you must assign the default code as instructed by the ICD-10-CM guidelines and the Alphabetic Index.
"""

ENTITY_EXTRACTION_PROMPT = """
Read the following clinical note and extract all medically relevant entities.
Specifically, look for:
1. Definitive Diagnoses (e.g., "Type 2 Diabetes Mellitus", "Essential Hypertension")
2. Symptoms / Signs (e.g., "chest pain", "shortness of breath") - NOTE: Only list these if they are not explicitly linked to a definitive diagnosis.
3. Pathogens / Organisms (e.g., "Staphylococcus aureus", "E. coli")

Return your extraction as a structured JSON list containing objects with the keys:
- entity_name: The name of the condition, symptom, or pathogen.
- entity_type: One of "diagnosis", "symptom", or "pathogen".
- text_span: The exact text from the note that justifies this extraction.

Clinical Note:
{clinical_note}
"""

RECONCILIATION_PROMPT = """
You are performing a medical coding reconciliation step. 
You will be provided with:
1. A list of medical entities extracted from a clinical note.
2. A list of ICD-10-CM codes that have *already been assigned* to this patient encounter.

Your task is to identify any MISSING codes (gaps) based on the extracted entities.
Cross-reference the extracted entities with the already assigned codes. 
If an extracted definitive diagnosis or necessary supplemental code (like a pathogen) is not represented in the assigned codes, suggest the appropriate ICD-10-CM code to fill the gap.

For each missing code you suggest, provide:
1. suggested_code: The ICD-10-CM code you are proposing.
2. justification_span: The text span from the clinical note justifying this addition.
3. guideline_rationale: The specific ICD-10-CM guideline or rule that makes this code necessary.

Extracted Entities:
{extracted_entities}

Already Assigned Codes:
{assigned_codes}
"""

## Business Logic Rules

In [ ]:
_RULES_DB = None

def get_rules_db():
    global _RULES_DB
    if _RULES_DB is None:
        base_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
        rules_path = os.path.join(base_dir, 'assets', 'ICD10_Assets', 'rules.json')
        try:
            with open(rules_path, 'r') as f:
                _RULES_DB = json.load(f)
        except FileNotFoundError:
            _RULES_DB = {}
    return _RULES_DB

def _check_conflict(c1: str, c2: str, rule_type: str) -> str | None:
    rules = get_rules_db()
    seen_prefixes = set()
    for prefix_len in range(len(c1), 2, -1):
        prefix = c1[:prefix_len].rstrip('.')
        if prefix in seen_prefixes:
            continue
        seen_prefixes.add(prefix)
        if prefix in rules:
            for excluded in rules[prefix].get(rule_type, []):
                if c2.startswith(excluded):
                    return f"Code {c1} has an {rule_type} note for {excluded} which conflicts with {c2}."
    return None

def check_excludes1(primary_code: str, additional_code: str) -> dict:
    """
    Checks if there's an Excludes1 conflict between two codes.
    """
    conflict_msg = _check_conflict(primary_code, additional_code, "excludes1") or _check_conflict(additional_code, primary_code, "excludes1")
    if conflict_msg:
        return {"valid": False, "reason": conflict_msg, "rule_type": "Excludes1"}
    return {"valid": True, "reason": "No Excludes1 conflict found.", "rule_type": "Excludes1"}

def check_code_first(codes: list[str]) -> list[dict]:
    results = []
    rules = get_rules_db()
    for i, code in enumerate(codes):
        seen_prefixes = set()
        for prefix_len in range(len(code), 2, -1):
            prefix = code[:prefix_len].rstrip('.')
            if prefix in seen_prefixes:
                continue
            seen_prefixes.add(prefix)
            if prefix in rules:
                for cf in rules[prefix].get("codeFirst", []):
                    found_index = -1
                    for j, assigned_code in enumerate(codes):
                        if assigned_code.startswith(cf):
                            found_index = j
                            break
                    if found_index == -1:
                        results.append({
                            "valid": False, 
                            "reason": f"Code {code} has a 'Code First' rule for {cf}. The underlying condition ({cf}) is missing from the assigned codes.",
                            "rule_type": "CodeFirst"
                        })
                    elif found_index > i:
                        results.append({
                            "valid": False, 
                            "reason": f"Code {code} has a 'Code First' rule for {cf}. The underlying condition ({codes[found_index]}) must be sequenced BEFORE {code}.",
                            "rule_type": "CodeFirst"
                        })
    return results

def check_use_additional_code(codes: list[str]) -> list[dict]:
    results = []
    rules = get_rules_db()
    for code in codes:
        seen_prefixes = set()
        for prefix_len in range(len(code), 2, -1):
            prefix = code[:prefix_len].rstrip('.')
            if prefix in seen_prefixes:
                continue
            seen_prefixes.add(prefix)
            if prefix in rules:
                for uac in rules[prefix].get("useAdditionalCode", []):
                    found = any(assigned_code.startswith(uac) for assigned_code in codes)
                    if not found:
                        results.append({
                            "valid": False,
                            "reason": f"Code {code} has a 'Use Additional Code' rule for {uac}, which is missing from the assigned codes.",
                            "rule_type": "UseAdditionalCode"
                        })
    return results

def check_7th_character(codes: list[str]) -> list[dict]:
    results = []
    rules = get_rules_db()
    for code in codes:
        clean_code = code.replace(".", "")
        for prefix_len in range(len(code), 2, -1):
            prefix = code[:prefix_len].rstrip('.')
            if prefix in rules and rules[prefix].get("requires_7th_char", False):
                if len(clean_code) < 7:
                    results.append({
                        "valid": False,
                        "reason": f"Code {code} requires a 7th character extension. If the base code is less than 6 characters, you must use placeholder 'X' to reach the 7th character.",
                        "rule_type": "7th_character_required"
                    })
                break
    return results

def check_coding_rules(codes: list[str]) -> list[dict]:
    """
    Validates a list of ICD-10-CM codes against coding conventions.
    """
    results = []
    
    # 1. Check for Excludes1 conflicts
    for i, code1 in enumerate(codes):
        for code2 in codes[i+1:]:
            res = check_excludes1(code1, code2)
            if not res["valid"]:
                results.append(res)
                
    # 2. Check Code First rules
    results.extend(check_code_first(codes))
    
    # 3. Check Use Additional Code rules
    results.extend(check_use_additional_code(codes))
    
    # 4. Check 7th character requirements
    results.extend(check_7th_character(codes))

    if not results:
        results.append({"valid": True, "reason": "All codes passed basic validation.", "rule_type": "all"})
        
    return results

## MCP Tools (Converted to Native Functions)

In [ ]:
# We will store the vector database in a local directory.
# In Kaggle, this will be inside /kaggle/working/
DB_DIR = os.getenv("CHROMA_DB_DIR", "./chroma_db")

def get_chroma_client():
    """Initializes and returns the ChromaDB client."""
    return chromadb.PersistentClient(path=DB_DIR)

def get_icd10_collection():
    """Retrieves or creates the ICD-10 collection."""
    client = get_chroma_client()
    return client.get_or_create_collection(name="icd10_index")

def search_icd10_rag(query: str, n_results: int = 5) -> list[dict]:
    """
    Search the ICD-10-CM guidelines and tabular list for a given query.
    
    Args:
        query: The medical term or symptom to search for.
        n_results: Number of results to return.
        
    Returns:
        List of matching ICD-10 codes and descriptions.
    """
    collection = get_icd10_collection()
    
    # In a real scenario with populated data, we query the collection.
    # We use a try-except to handle cases where the collection is empty gracefully.
    try:
        results = collection.query(
            query_texts=[query],
            n_results=n_results
        )
        
        # Format results into a list of dicts
        formatted_results = []
        if results and "documents" in results and results["documents"]:
            for i in range(len(results["documents"][0])):
                doc = results["documents"][0][i]
                meta = results["metadatas"][0][i] if "metadatas" in results and results["metadatas"] else {}
                formatted_results.append({
                    "code": meta.get("code", "UNKNOWN"),
                    "description": doc,
                    "distance": results["distances"][0][i] if "distances" in results else None
                })
        return formatted_results
    except Exception as e:
        return [{"error": str(e), "message": "Failed to query ChromaDB or collection is empty."}]

def add_icd10_documents(documents: list[str], metadatas: list[dict], ids: list[str]):
    """
    Helper function to populate the ChromaDB vector store.
    """
    collection = get_icd10_collection()
    collection.add(
        documents=documents,
        metadatas=metadatas,
        ids=ids
    )



logger = logging.getLogger(__name__)

def query_umls(term: str) -> dict:
    """
    Query the UMLS API to normalize medical jargon or symptoms.
    
    Args:
        term: The medical term to normalize.
        
    Returns:
        A dictionary containing the normalized concept and details.
    """
    # Use KaggleSecrets if on Kaggle, otherwise use os.getenv
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        api_key = user_secrets.get_secret("UMLS_API_KEY")
    except ImportError:
        api_key = os.getenv("UMLS_API_KEY")
    
    # PLACEHOLDER LOGIC: Currently waiting for UMLS API approval
    # We will return mocked responses for demonstration purposes
    logger.info(f"Querying UMLS placeholder for term: {term}")
    
    term_lower = term.lower()
    
    # Simple mocked database for the demo
    mock_db = {
        "heart attack": {
            "concept_id": "C0027051",
            "name": "Myocardial Infarction",
            "semantic_type": "Disease or Syndrome",
            "is_definitive": True
        },
        "headache": {
            "concept_id": "C0018681",
            "name": "Headache",
            "semantic_type": "Sign or Symptom",
            "is_definitive": False
        },
        "type 2 diabetes": {
            "concept_id": "C0011860",
            "name": "Diabetes Mellitus, Non-Insulin-Dependent",
            "semantic_type": "Disease or Syndrome",
            "is_definitive": True
        }
    }
    
    for key, value in mock_db.items():
        if key in term_lower or term_lower in key:
            return value
            
    return {
        "concept_id": "UNKNOWN",
        "name": term,
        "semantic_type": "Unknown",
        "message": "Term not found in mock DB. Real API would query UMLS here."
    }


# Native Tools for Gemini
def search_icd10_tool(query: str, top_k: int = 5) -> str:
    """Search the ICD-10-CM guidelines and tabular list for a given query."""
    return json.dumps(search_icd10_rag(query, n_results=top_k), indent=2)

def normalize_medical_term_tool(term: str) -> str:
    """Query the UMLS API to normalize medical jargon or symptoms."""
    return json.dumps(query_umls(term), indent=2)

def validate_icd10_codes_tool(codes: list[str]) -> str:
    """Validates a list of ICD-10-CM codes against Excludes1, Excludes2, and 7th character rules."""
    return json.dumps(check_coding_rules(codes), indent=2)

GEMINI_TOOLS = [search_icd10_tool, normalize_medical_term_tool, validate_icd10_codes_tool]

## Evaluation Pipeline

In [ ]:
# Data models for structured output
class Entity(BaseModel):
    entity_name: str
    entity_type: str
    text_span: str

class ExtractionResponse(BaseModel):
    entities: list[Entity]

class MissingCode(BaseModel):
    suggested_code: str
    justification_span: str
    guideline_rationale: str

class ReconciliationResponse(BaseModel):
    missing_codes: list[MissingCode]

def load_eval_data():
    diag_df = load_patient_diagnoses(os.path.join(DATA_PATH, 'diagnoses.csv'))
    raw_diag = pd.read_csv(os.path.join(DATA_PATH, 'diagnoses.csv'))
    merged_diag = raw_diag.merge(diag_df[['patient_id', 'visit_date', 'assigned_icd10_codes']], on=['patient_id', 'visit_date'])
    patients_df = pd.read_csv(os.path.join(DATA_PATH, 'patients.csv'))
    meds_df = pd.read_csv(os.path.join(DATA_PATH, 'medications.csv'))
    return merged_diag, patients_df, meds_df

def generate_synthetic_note(patient: pd.Series, diag_row: pd.Series, meds: pd.DataFrame) -> str:
    age = patient['age']
    sex = "male" if patient['sex'] == "M" else "female"
    smoking = patient['smoking_status']
    
    primary = str(diag_row['primary_diagnosis']).replace('_', ' ')
    secondary = str(diag_row['secondary_diagnoses']).replace('|', ', ').replace('_', ' ')
    if pd.isna(diag_row['secondary_diagnoses']) or not secondary or secondary.lower() == 'nan':
        secondary_str = "No secondary conditions reported."
    else:
        secondary_str = f"Secondary conditions include {secondary}."
        
    visit_type = diag_row['visit_type']
    
    meds_str = ""
    if not meds.empty:
        med_lines = []
        for _, m in meds.iterrows():
            med_lines.append(f"{m['medication']} {m['dose']} {m['unit']} {m['frequency']}")
        meds_str = "Patient is currently prescribed: " + ", ".join(med_lines) + "."
    
    note = f"A {age}-year-old {sex} {smoking} smoker presented for a {visit_type} visit. "
    note += f"Primary diagnosis: {primary}. {secondary_str} "
    if meds_str:
        note += f" {meds_str}"
        
    return note

def evaluate_pipeline(n_samples=2):
    merged_diag, patients_df, meds_df = load_eval_data()
    valid_encounters = merged_diag[merged_diag['assigned_icd10_codes'].apply(len) >= 2]
    sampled = valid_encounters.sample(n=n_samples, random_state=42)
    
    results = []
    
    for idx, diag_row in sampled.iterrows():
        pid = diag_row['patient_id']
        patient = patients_df[patients_df['patient_id'] == pid].iloc[0]
        meds = meds_df[meds_df['patient_id'] == pid]
        
        note = generate_synthetic_note(patient, diag_row, meds)
        true_codes = diag_row['assigned_icd10_codes']
        
        dropped_code = random.choice(true_codes)
        assigned_codes = [c for c in true_codes if c != dropped_code]
        
        print(f"\n--- Evaluating Patient {pid} ---")
        print(f"Assigned (Provided to Agent): {assigned_codes}")
        print(f"Dropped (Target for Agent): {dropped_code}")
        
        # 1. Entity Extraction
        prompt = ENTITY_EXTRACTION_PROMPT.format(clinical_note=note)
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=genai.types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                response_mime_type="application/json",
                response_schema=ExtractionResponse,
                temperature=0.0
            )
        )
        extracted_entities = response.text
        
        # 2. Reconciliation with Tools
        recon_prompt = RECONCILIATION_PROMPT.format(
            extracted_entities=extracted_entities,
            assigned_codes=assigned_codes
        )
        recon_response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=recon_prompt,
            config=genai.types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                response_mime_type="application/json",
                response_schema=ReconciliationResponse,
                temperature=0.0
            )
        )
        
        try:
            missing = json.loads(recon_response.text).get("missing_codes", [])
        except Exception:
            missing = []
            
        suggested_codes = [m["suggested_code"] for m in missing]
        found = dropped_code in suggested_codes
        print(f"Agent Suggested: {suggested_codes} | Success: {found}")
        
        results.append({
            "found": found,
            "suggested_codes": suggested_codes
        })
        time.sleep(4)
        
    # Metrics
    total = len(results)
    successes = sum([r["found"] for r in results])
    print(f"\nEvaluation Complete! Recall: {successes}/{total} ({(successes/total)*100:.1f}%)")
    
    return results

results = evaluate_pipeline(2)
